# Виды частиц и реакции

В FESTIM каждая концентрация связана с определённым видом частиц (`Species`). Реакции между видами частиц задаются объектами `Reaction`.

В этом разделе достаточно понять три конструкции:

- мобильные и неподвижные виды частиц;
- неявные виды частиц;
- объёмные реакции между ними.

## Явные виды частиц

Концентрация явного вида частиц является неизвестной величиной задачи и рассчитывается FESTIM.

Мобильные частицы диффундируют в материале. Для неподвижных частиц диффузионный член отсутствует. Аргумент `mobile=True` используется по умолчанию.

In [9]:
import festim as F

mobile_H = F.Species(
    name="H",
)

trapped_H = F.Species(
    name="H_trapped",
    mobile=False,
)

model = F.HydrogenTransportProblem()
model.species = [mobile_H, trapped_H]

## Неявные виды частиц

Иногда концентрацию одного вида можно выразить через другие концентрации. Например, концентрация свободных ловушек равна

$$
n_{\mathrm{empty}} = n_{\mathrm{total}} - c_{\mathrm{trapped}}.
$$

В таком случае свободные ловушки можно задать как `ImplicitSpecies`. Для них не решается отдельное уравнение, поэтому они не добавляются в `model.species`.

`ImplicitSpecies` не подходит для мобильных частиц.


In [10]:
empty_traps = F.ImplicitSpecies(
    name="empty_traps",
    n=2.0,
    others=[trapped_H],
)

# В список model.species входят только явные виды частиц
model.species = [mobile_H, trapped_H]

**Примечание.** В качестве "частиц" также могут рассматриваться межузельные атомы рассматриваемого материала или примесей (interstitials), перенос которых также можно моделировать.

## Реакции

Объект `Reaction` связывает реагенты и продукты реакции. Скорости прямой и обратной реакций задаются законами Аррениуса:

$$
k(T) = k_0 \exp\left(-\frac{E_k}{k_B T}\right),
\qquad
p(T) = p_0 \exp\left(-\frac{E_p}{k_B T}\right).
$$

Рассмотрим захват мобильного водорода свободной ловушкой:

$$
\mathrm{H + [\,] \rightleftharpoons [H]}.
$$


In [11]:
import numpy as np

model = F.HydrogenTransportProblem()

model.mesh = F.Mesh1D(
    vertices=np.linspace(0.0, 1e-3, 101),
)

material = F.Material(
    D_0=1.11e-6,
    E_D=0.4,
)

volume = F.VolumeSubdomain1D(
    id=1,
    borders=[0.0, 1e-3],
    material=material,
)

mobile_H = F.Species("H")
trapped_H = F.Species("H_trapped", mobile=False)

# Число свободных ловушек вычисляется как
# n_total - c_trapped
empty_traps = F.ImplicitSpecies(
    name="empty_traps",
    n=2.0,
    others=[trapped_H],
)

model.species = [mobile_H, trapped_H]
model.subdomains = [volume]

model.reactions = [
    F.Reaction(
        reactant=[mobile_H, empty_traps],
        product=[trapped_H],
        k_0=0.01,
        E_k=0.0,
        p_0=0.1,
        E_p=0.0,
        volume=volume,
    )
]


## Упрощённое задание ловушки

Для простого случая — один мобильный вид частиц и одна частица в каждой ловушке — можно использовать класс `Trap`. Он задаёт ту же физику более компактно, но предоставляет меньше гибкости, чем общий объект `Reaction`.

In [12]:
model = F.HydrogenTransportProblem()

model.mesh = F.Mesh1D(
    vertices=np.linspace(0.0, 1e-3, 101),
)

material = F.Material(
    D_0=1.11e-6,
    E_D=0.4,
)

volume = F.VolumeSubdomain1D(
    id=1,
    borders=[0.0, 1e-3],
    material=material,
)

mobile_H = F.Species("H")

model.species = [mobile_H]
model.subdomains = [volume]

model.traps = [
    F.Trap(
        mobile_species=mobile_H,
        k_0=0.01,
        E_k=0.0,
        p_0=0.1,
        E_p=0.0,
        volume=volume,
        n=2.0,
        name="Ловушка"
    )
]


## Дополнительные примеры

Следующие примеры показывают, что более сложные схемы собираются из тех же объектов `Species`, `ImplicitSpecies` и `Reaction`.

### Одна ловушка и два изотопа

Водород и тритий конкурируют за один общий набор ловушек:

$$
\mathrm{H + [\,] \rightleftharpoons [H]},
\qquad
\mathrm{T + [\,] \rightleftharpoons [T]}.
$$


In [13]:
H = F.Species("H")
T = F.Species("T")

trapped_H = F.Species("H_trapped", mobile=False)
trapped_T = F.Species("T_trapped", mobile=False)

# Общая концентрация свободных ловушек
empty_traps = F.ImplicitSpecies(
    n=2.0,
    others=[trapped_H, trapped_T],
)

trapping_H = F.Reaction(
    reactant=[H, empty_traps],
    product=[trapped_H],
    k_0=0.1,
    E_k=0.0,
    p_0=0.001,
    E_p=0.0,
    volume=volume,
)

trapping_T = F.Reaction(
    reactant=[T, empty_traps],
    product=[trapped_T],
    k_0=0.1,
    E_k=0.0,
    p_0=0.001,
    E_p=0.0,
    volume=volume,
)

reactions_two_isotopes = [trapping_H, trapping_T]

### Множественный захват одного изотопа

Одна ловушка может последовательно удерживать несколько атомов:

$$
\mathrm{H + [\,] \rightleftharpoons [H]},
\qquad
\mathrm{H + [H] \rightleftharpoons [HH]}.
$$

In [14]:
H = F.Species("H")

trapped_1H = F.Species("1H_trapped", mobile=False)
trapped_2H = F.Species("2H_trapped", mobile=False)

empty_traps = F.ImplicitSpecies(
    n=2.0,
    others=[trapped_1H, trapped_2H],
)

first_capture = F.Reaction(
    reactant=[H, empty_traps],
    product=[trapped_1H],
    k_0=0.01,
    E_k=0.0,
    p_0=0.1,
    E_p=0.0,
    volume=volume,
)

second_capture = F.Reaction(
    reactant=[H, trapped_1H],
    product=[trapped_2H],
    k_0=0.02,
    E_k=0.0,
    p_0=0.1,
    E_p=0.0,
    volume=volume,
)

reactions_multi_occupancy = [first_capture, second_capture]

### Радиоактивный распад трития

Реакция без продуктов используется как сток трития:

$$
\mathrm{T \rightarrow \varnothing}.
$$

In [15]:
T = F.Species("T")

decay_constant = 1.0  # 1/с

tritium_decay = F.Reaction(
    reactant=[T],
    k_0=decay_constant,
    E_k=0.0,
    volume=volume,
)